## Data loads

In [ ]:
import pandas as pd
import os
from glob import glob

from src.utils.preprocessing import Preprocessor

from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

from collections import Counter
from tqdm import tqdm
import numpy as np



In [ ]:

# font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
# font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
# font_fname = os.environ.get('PROJECT_FONT_PATH', '')
font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
# os.listdir(font_fname)

font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

# figdir = os.path.join('outputs','figures')
# os.makedirs(figdir, exist_ok = True)

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]
measure_dict = dict()

In [ ]:
raw_data_dir = glob(os.path.join('data','reports','*'))
print(raw_data_dir)
df = pd.read_csv(raw_data_dir[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


In [ ]:
preprocessor.

for gs, sdf in target_df.groupby(['병원등록번호','수술일자','수술실','진단명','수술명','검사접수일자','검사코드']):
    if sdf.shape[0] > 1:
        print(gs)
        print(sdf.shape)
    # break

### Rule based dataframe

In [ ]:
rule_path_dir = sorted(glob(os.path.join('data','rule_labels', '*labeled.csv')))
print(rule_path_dir)
rule_meta_path, rule_recur_path = rule_path_dir
rule_dict = dict(
    metas = pd.read_csv(rule_meta_path, encoding = 'utf-8-sig', engine = 'c'),
    recur = pd.read_csv(rule_recur_path, encoding = 'utf-8-sig', engine = 'c')
)


### 1st revision dataframe

In [ ]:
# reviewer gold workbooks
# metastasis_positive
revision_01_path_dir = sorted(glob(os.path.join('data','reviewer', '*positive.xlsx')))
print(revision_01_path_dir)
rev_01_meta_path, rev_01_recur_path = revision_01_path_dir
revision_dict = dict()
revision_dict[1] = dict(
    metas = pd.read_excel(rev_01_meta_path, engine = 'openpyxl').dropna(),
    recur = pd.read_excel(rev_01_recur_path, engine = 'openpyxl').dropna()
)


In [ ]:
revision_dict[1]['recur']['label'].value_counts(), revision_dict[1]['metas']['label'].value_counts(),

### 2nd revision dataframe

In [ ]:
revision_02_meta_path_dir = sorted(glob(os.path.join('data','reviewer', '*meta*.xlsx')))
revision_02_recur_path_dir = sorted(glob(os.path.join('data','reviewer', '*recur*.xlsx')))

print(revision_02_meta_path_dir)
print(revision_02_recur_path_dir)
revision_dict[2] = dict(
    metas = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_meta_path_dir]),
    recur = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_recur_path_dir])
)

In [ ]:
revision_dict[1]['recur']['label'].value_counts(), revision_dict[1]['metas']['label'].value_counts(),

In [ ]:
revision_dict[2]['recur']['재분류'].value_counts(), revision_dict[2]['metas']['재분류'].value_counts(),

## Figure

### Def function

In [ ]:
figure_dir = os.path.join('outputs','figures')
os.makedirs(figure_dir, exist_ok = True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
preprocessor.df

In [ ]:
l = df['검사결과결론내용'].str.len()

metas_l = preprocessor.df.apply(lambda x: '\n'.join(preprocessor.extract_parts(x, 'metas'))).str.len()
diff_metas_l = l - metas_l

recur_l = preprocessor.df.apply(lambda x: '\n'.join(preprocessor.extract_parts(x, 'recur'))).str.len()
diff_recur_l = l - recur_l

print(metas_l.shape, recur_l.shape)

In [ ]:
import re
t = 'metas'
pattern = re.compile(r'\s'+t)
matches = preprocessor.df[preprocessor.df.apply(lambda x: bool(pattern.search(x)))]
pattern = re.compile(t)
matches2 = preprocessor.df[preprocessor.df.apply(lambda x: bool(pattern.search(x)))]
matches.shape, matches2.shape

In [ ]:

metas_l = preprocessor.df.apply(lambda x: '\n'.join(preprocessor.extract_parts(x, 'metas'))).str.len()
diff_metas_l = l - metas_l

recur_l = preprocessor.df.apply(lambda x: '\n'.join(preprocessor.extract_parts(x, 'recur'))).str.len()
diff_recur_l = l - recur_l

print(metas_l.shape, recur_l.shape)

In [ ]:
plt.rcParams['font.size'] = 20
# fig, axes = plt.subplots(1, 2, figsize = (20, 5), sharey = True)
# x축 길이 비율이 3:1이 되도록 subplot의 width_ratios 조정
fig, axes = plt.subplots(1, 2, figsize = (20, 5), sharey = True, gridspec_kw={'width_ratios': [4, 1]})

box_width = 60
quant_l = (l // box_width) * box_width + box_width // 2

plot_df = pd.DataFrame().from_dict(
    dict(
        Length = pd.concat([quant_l, quant_l]),
        Diff = pd.concat([diff_recur_l, diff_metas_l]),
        Types = len(recur_l) * ['Recurrence'] + len(metas_l) * ['Metastasis']
    )
)

sns.boxplot(data = plot_df, x='Length', y='Diff', ax=axes[0], hue = 'Types')

# x tick label 가져와서 30도정도 회전시켜줍니다.
for tick in axes[0].get_xticklabels():
    tick.set_rotation(45)
    tick.set_ha('center')


axes[0].set_xlabel('Raw length')
axes[0].set_ylabel(r'$\Delta$Length')

sns.histplot(data = plot_df, y = 'Diff', bins = 50, ax = axes[1], hue = 'Types')
# d.rename(r'$\Delta$Length')
axes[1].set_xscale('log')
axes[1].set_xlabel('Count (log)')
fig.tight_layout()
# fig.savefig(os.path.join(figure_dir, 'length_distribution.pdf'), dpi = 400)


l1 = df['검사결과결론내용'].str.len()
l2 = preprocessor.df.str.len()

from transformers import AutoTokenizer

model_path = os.path.join('..','..','..','model','PubMedBERT-base-uncased-sts-combined')

tokenizer = AutoTokenizer.from_pretrained(model_path)

# 샘플이 많아서 배치 단위로 토크나이즈
def batch_tokenize(texts, batch_size=512):
    all_encodings = {k: [] for k in tokenizer.model_input_names}
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        encoding = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')

        # pad 토큰을 제외한 실제 토큰들만 각 field별로 모읍니다.
        # 각 배치별로 pad를 제외한 부분만 슬라이싱 후 extend
        input_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]

        for k in tokenizer.model_input_names:
            # attention_mask가 1인 부분만 남긴다
            for idx in range(input_ids.shape[0]):
                mask = attention_mask[idx].bool()
                # k에 해당하는 텐서에서 pad를 제외한 부분만 선택
                value = encoding[k][idx][mask]
                all_encodings[k].append(value)
        # for k in tokenizer.model_input_names:
        #     all_encodings[k].append(encoding[k])
    # Concatenate each field along the first dimension
    # for k in all_encodings:
    #     all_encodings[k] = torch.cat(all_encodings[k], dim=0)
    return all_encodings

import torch

tokens_l1 = batch_tokenize(df['검사결과결론내용'].tolist())
tokens_l2 = batch_tokenize(preprocessor.df.tolist())



token_l1 = [len(i) for i in tokens_l1['input_ids']]
token_l2 = [len(i) for i in tokens_l2['input_ids']]

fig, axes = plt.subplots(2, 1, figsize = (5, 8), sharex = True)
sns.histplot(token_l1, label="Before", linewidth=1, ax = axes[0], binwidth = 10, alpha = .5)
sns.histplot(token_l2, label="After", linewidth=1, ax = axes[0], binwidth = 10, color = 'tab:red', alpha = .5)
axes[0].set_yscale('log')


# l1, l2을 stack 형태로 하나의 데이터프레임으로 합치기
lengths_df = pd.DataFrame({
    # 'Length': pd.concat([token_l1, token_l2], ignore_index=True),
    'Length': token_l1 + token_l2,
    'Type': ['Before'] * len(token_l1) + ['After'] * len(token_l2)
})

sns.ecdfplot(
    data=lengths_df,
    x='Length',
    hue='Type',
    palette={'Before': 'tab:red', 'After': 'tab:blue'},
    ax=axes[1]
)
axes[1].set_xlabel('Tokenized Length')
axes[1].set_ylabel('Proportion')
# axes[1].set_xscale('log')
axes[0].legend(title = 'Preprocessing', labels = ['Before','After'])
axes[1].legend().remove()
# axes[1].set_xscale('log')
fig.tight_layout()
fig.savefig(os.path.join(figure_dir, 'tokenized_length_distribution.pdf'), dpi = 400)

In [ ]:
plt.rcParams['font.size'] = 20

# 2행 3열로 subplot grid 구성: (0,0:3)에는 boxplot, (1,0), (1,1), (1,2)에 각각 나머지 그림
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(22, 14))
gs = GridSpec(2, 3, figure=fig)

box_width = 60
l = df['검사결과결론내용'].str.len()
d = preprocessor.df.str.len() - l
quant_l = (l // box_width) * box_width + box_width // 2


plot_df = pd.DataFrame().from_dict(
    dict(
        Length = pd.concat([quant_l, quant_l]),
        Diff = pd.concat([diff_recur_l, diff_metas_l]),
        Types = len(recur_l) * ['Recurrence'] + len(metas_l) * ['Metastasis']
    )
)


# 첫 번째 행 (0, 0:3) - Boxplot of ΔLength by Raw Length (가로로 3칸 합치기)
ax_box = fig.add_subplot(gs[1, :])
sns.boxplot(data = plot_df, x='Length', y='Diff', ax=ax_box, hue = 'Types')
for tick in ax_box.get_xticklabels():
    tick.set_rotation(45)
    tick.set_ha('center')
ax_box.set_xlabel('Raw conclusion')
ax_box.set_ylabel(r'$\Delta$Length')
ax_box.set_title(r'$\Delta$Length from raw conclusion')

# 두 번째 행 (1, 0): ΔLength의 y축 히스토그램
ax_hist = fig.add_subplot(gs[0, 0])
# sns.histplot(data = plot_df, y = 'Diff', bins = 50, ax = axes[1], hue = 'Types')
sns.histplot(data = plot_df, x = 'Diff', bins = 50, hue = 'Types', ax = ax_hist)
ax_hist.set_yscale('log')
ax_hist.set_ylabel('Count')
ax_hist.set_xlabel(r'$\Delta$Length')
ax_hist.set_title(r'$\Delta$Length histogram')

# 두 번째 행 (1, 1): Tokenized Length ECDF
ax_ecdf = fig.add_subplot(gs[0, 1])
lengths_df = pd.DataFrame({
    'Length': pd.concat([recur_l, metas_l, l], axis = 0) ,
    'Type': ['Recurrence'] * len(recur_l) + ['Metastasis'] * len(metas_l) + ['Raw'] * len(l)
})
sns.ecdfplot(
    data=lengths_df,
    x='Length',
    hue='Type',
    palette={ 'Recurrence': 'tab:blue', 'Metastasis': 'tab:orange', 'Raw': 'Gray',},
    ax=ax_ecdf
)
ax_ecdf.set_xlabel('Length of conclusion')
ax_ecdf.set_ylabel('Proportion')
# ax_ecdf.legend(title = 'Preprocessing', labels = ['Before','After'])
ax_ecdf.set_title('ECDF of conclusion')
# ax_ecdf.set_xscale('log')

# # (1,2)는 비워두거나, 필요하다면 다른 시각화 추가 가능

ax_hist = fig.add_subplot(gs[0, 2])
hist_bin_width = 30
sns.histplot(l, label="Raw", linewidth=1, ax = ax_hist, binwidth = hist_bin_width, color = 'gray', alpha = .4, binrange=(0, l.max()))
sns.histplot(metas_l, label="Metastasis", linewidth=1, ax = ax_hist, binwidth = hist_bin_width, color = 'tab:orange', alpha = .4, binrange=(0, l.max()))
sns.histplot(recur_l, label="Recurrence", linewidth=1, ax = ax_hist, binwidth = hist_bin_width, color = 'tab:blue', alpha = .4, binrange=(0, l.max()))

ax_hist.set_yscale('log')
ax_hist.set_xlabel('Length of conclusion')
legend = ax_hist.get_legend_handles_labels()
ax_hist.legend(legend[0][::-1], legend[1][::-1], title = 'Preprocessing')
ax_hist.set_title('Histogram of conclusion')
fig.tight_layout()
fig.savefig(os.path.join(figure_dir, 'distribution_summary.pdf'), dpi = 400)

# fig.tight_layout()

In [ ]:
plt.rcParams['font.size'] = 20

# 2행 3열로 subplot grid 구성: (0,0:3)에는 boxplot, (1,0), (1,1), (1,2)에 각각 나머지 그림
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 5))
gs = GridSpec(1, 3, figure=fig)

box_width = 60
l = df['검사결과결론내용'].str.len()
d = preprocessor.df.str.len() - l
quant_l = (l // box_width) * box_width + box_width // 2


plot_df = pd.DataFrame().from_dict(
    dict(
        Length = pd.concat([quant_l, quant_l]),
        Diff = pd.concat([diff_recur_l, diff_metas_l]),
        Types = len(recur_l) * ['Recurrence'] + len(metas_l) * ['Metastasis']
    )
)

# 두 번째 행 (1, 0): ΔLength의 y축 히스토그램
ax_hist = fig.add_subplot(gs[0, 0])
# sns.histplot(data = plot_df, y = 'Diff', bins = 50, ax = axes[1], hue = 'Types')
sns.histplot(data = plot_df, x = 'Diff', bins = 50, hue = 'Types', ax = ax_hist)
ax_hist.set_yscale('log')
ax_hist.set_ylabel('Count')
ax_hist.set_xlabel(r'$\Delta$Length')
ax_hist.set_title(r'$\Delta$Length histogram')

# 두 번째 행 (1, 1): Tokenized Length ECDF
ax_ecdf = fig.add_subplot(gs[0, 1])
lengths_df = pd.DataFrame({
    'Length': pd.concat([recur_l, metas_l, l], axis = 0) ,
    'Type': ['Recurrence'] * len(recur_l) + ['Metastasis'] * len(metas_l) + ['Raw'] * len(l)
})

# ECDF plot
sns.ecdfplot(
    data=lengths_df,
    x='Length',
    hue='Type',
    palette={ 'Recurrence': 'tab:blue', 'Metastasis': 'tab:orange', 'Raw': 'Gray',},
    ax=ax_ecdf
)

ax_ecdf.set_xlabel('Length of report')
ax_ecdf.set_ylabel('Proportion')
ax_ecdf.legend(
    ['Recurrence','Metastasis','Raw'],
    loc = 'lower right'
)
ax_ecdf.set_title('ECDF of report')
ax_ecdf.set_xscale('log')



# 두 번째 행 (1, 1): Tokenized Length ECDF
ax_case_ecdf = fig.add_subplot(gs[0, 2])

recur_loc = recur_l > 0
case_recur_l = recur_l.loc[recur_loc]
case_recur_raw_l = l.loc[recur_loc]

metas_loc = metas_l > 0
case_metas_l = metas_l.loc[metas_loc]
case_metas_raw_l = l.loc[metas_loc]

case_lengths_df = pd.DataFrame({
    'Length': pd.concat([
        case_recur_l,
        case_recur_raw_l,  
        case_metas_l,
        case_metas_raw_l, 
    ], axis = 0) ,
    'Type': ['Recurrence'] * len(case_recur_l) + \
            ['Raw Recurrence'] * len(case_recur_raw_l) + \
            ['Metastasis'] * len(case_metas_l) + \
            ['Raw Metastasis'] * len(case_metas_raw_l)
})



# ECDF plot
# Note: sns.ecdfplot의 linestyle에는 dict가 아닌 str만 허용됨.
# 스타일을 각 곡선마다 다르게 하려면 별도 플롯 반복 또는 한 번에 하나씩 호출 필요
palette = {
    'Recurrence': 'tab:blue',
    'Raw Recurrence': 'tab:blue',
    'Metastasis': 'tab:orange',
    'Raw Metastasis': 'tab:orange'
}
# 기본(ecdfplot) 에선 모든 곡선에 동일 line 스타일만 가능. 별개로 그릴 때만 다름
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Recurrence'],
    x='Length',
    ax=ax_case_ecdf,
    label='Recurrence',
    color='tab:blue',
    linestyle='-'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Raw Recurrence'],
    x='Length',
    ax=ax_case_ecdf,
    label='Raw Recurrence',
    color='tab:blue',
    linestyle='--'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Metastasis'],
    x='Length',
    ax=ax_case_ecdf,
    label='Metastasis',
    color='tab:orange',
    linestyle='-'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Raw Metastasis'],
    x='Length',
    ax=ax_case_ecdf,
    label='Raw Metastasis',
    color='tab:orange',
    linestyle='--'
)
ax_case_ecdf.legend(['Rec.','Raw Rec.','Met.','Raw Met.'], loc='lower right')



ax_case_ecdf.set_xlabel('Length of report')
ax_case_ecdf.set_ylabel('Proportion')
ax_case_ecdf.set_title('ECDF of report by each Target')
ax_case_ecdf.set_xscale('log')

fig.tight_layout()
fig.savefig(os.path.join(figure_dir, 'distribution_summary_subset.png'), dpi = 400)


In [ ]:
plt.rcParams['font.size'] = 20

# 2행 3열로 subplot grid 구성: (0,0:3)에는 boxplot, (1,0), (1,1), (1,2)에 각각 나머지 그림
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(12, 5))
gs = GridSpec(1, 2, figure=fig)

box_width = 60
l = df['검사결과결론내용'].str.len()
d = preprocessor.df.str.len() - l
quant_l = (l // box_width) * box_width + box_width // 2


# plot_df = pd.DataFrame().from_dict(
#     dict(
#         Length = pd.concat([quant_l, quant_l]),
#         Diff = pd.concat([diff_recur_l, diff_metas_l]),
#         Types = len(recur_l) * ['Recurrence'] + len(metas_l) * ['Metastasis']
#     )
# )

# # 두 번째 행 (1, 0): ΔLength의 y축 히스토그램
# ax_hist = fig.add_subplot(gs[0, 0])
# # sns.histplot(data = plot_df, y = 'Diff', bins = 50, ax = axes[1], hue = 'Types')
# sns.histplot(data = plot_df, x = 'Diff', bins = 50, hue = 'Types', ax = ax_hist)
# ax_hist.set_yscale('log')
# ax_hist.set_ylabel('Count')
# ax_hist.set_xlabel(r'$\Delta$Length')
# ax_hist.set_title(r'$\Delta$Length histogram')

# 두 번째 행 (1, 1): Tokenized Length ECDF
ax_ecdf = fig.add_subplot(gs[0, 0])
lengths_df = pd.DataFrame({
    'Length': pd.concat([recur_l, metas_l, l], axis = 0) ,
    'Type': ['Recurrence'] * len(recur_l) + ['Metastasis'] * len(metas_l) + ['Raw'] * len(l)
})

# ECDF plot
sns.ecdfplot(
    data=lengths_df,
    x='Length',
    hue='Type',
    palette={ 'Recurrence': 'tab:blue', 'Metastasis': 'tab:orange', 'Raw': 'Gray',},
    ax=ax_ecdf
)

ax_ecdf.set_xlabel('Length of report')
ax_ecdf.set_ylabel('Proportion')
ax_ecdf.legend(
    ['Recurrence','Metastasis','Raw'],
    loc = 'lower right'
)
ax_ecdf.set_title('ECDF of report')
ax_ecdf.set_xscale('log')



# 두 번째 행 (1, 1): Tokenized Length ECDF
ax_case_ecdf = fig.add_subplot(gs[0, 1])

recur_loc = recur_l > 0
case_recur_l = recur_l.loc[recur_loc]
case_recur_raw_l = l.loc[recur_loc]

metas_loc = metas_l > 0
case_metas_l = metas_l.loc[metas_loc]
case_metas_raw_l = l.loc[metas_loc]

case_lengths_df = pd.DataFrame({
    'Length': pd.concat([
        case_recur_l,
        case_recur_raw_l,  
        case_metas_l,
        case_metas_raw_l, 
    ], axis = 0) ,
    'Type': ['Recurrence'] * len(case_recur_l) + \
            ['Raw Recurrence'] * len(case_recur_raw_l) + \
            ['Metastasis'] * len(case_metas_l) + \
            ['Raw Metastasis'] * len(case_metas_raw_l)
})



# ECDF plot
# Note: sns.ecdfplot의 linestyle에는 dict가 아닌 str만 허용됨.
# 스타일을 각 곡선마다 다르게 하려면 별도 플롯 반복 또는 한 번에 하나씩 호출 필요
palette = {
    'Recurrence': 'tab:blue',
    'Raw Recurrence': 'tab:blue',
    'Metastasis': 'tab:orange',
    'Raw Metastasis': 'tab:orange'
}
# 기본(ecdfplot) 에선 모든 곡선에 동일 line 스타일만 가능. 별개로 그릴 때만 다름
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Recurrence'],
    x='Length',
    ax=ax_case_ecdf,
    label='Recurrence',
    color='tab:blue',
    linestyle='-'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Raw Recurrence'],
    x='Length',
    ax=ax_case_ecdf,
    label='Raw Recurrence',
    color='tab:blue',
    linestyle='--'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Metastasis'],
    x='Length',
    ax=ax_case_ecdf,
    label='Metastasis',
    color='tab:orange',
    linestyle='-'
)
sns.ecdfplot(
    data=case_lengths_df[case_lengths_df['Type'] == 'Raw Metastasis'],
    x='Length',
    ax=ax_case_ecdf,
    label='Raw Metastasis',
    color='tab:orange',
    linestyle='--'
)
ax_case_ecdf.legend(['Rec.','Raw Rec.','Met.','Raw Met.'], loc='lower right')



ax_case_ecdf.set_xlabel('Length of report')
ax_case_ecdf.set_ylabel('Proportion')
ax_case_ecdf.set_title('ECDF of report by each Target')
ax_case_ecdf.set_xscale('log')

fig.tight_layout()
fig.savefig(os.path.join(figure_dir, 'new_distribution_summary_subset.png'), dpi = 400)


In [ ]:
figure_dir

In [ ]:
target_l = recur_l

def compute_ratio(target_l):
    target_loc = target_l > 0
    raw_l = preprocessor.df.loc[target_loc].str.len()
    avg_diff_token = (raw_l - target_l.loc[target_loc]).mean()
    avg_diff_ratio = avg_diff_token / raw_l.mean() * 100
    return (target_l == 0).mean() * 100, avg_diff_token, avg_diff_ratio

print('Recurrence')
print([i.round(2) for i in compute_ratio(recur_l)])

print('Metastasis')
print([i.round(2) for i in compute_ratio(metas_l)])